# Understanding Cat and Dog Predictions Using Grad-CAM
IEEE VIT Round 2 • Python / PyTorch • Pretrained ResNet-18

**Goal:** classify a balanced subset of the specified Kaggle dataset, inspect correct and incorrect predictions, and explain the model's spatial evidence using Grad-CAM. No training or fine-tuning is performed.

**Run:** In Google Colab choose Runtime → Change runtime type → T4 GPU (optional), then Runtime → Run all. The dataset download is approximately 756 MiB. CPU also works. The final cells display and export the actual results.

The bundled output files were generated on CPU outside Colab. Run this notebook in Colab and save it with its outputs for your own submission record.


## 1 Install the small additional dependencies
Colab supplies compatible PyTorch and Torchvision packages. We do not reinstall them. KaggleHub downloads the exact dataset version. If authentication is requested, use `kagglehub.login()` in a separate cell; never put credentials into the notebook or GitHub.

In [ ]:
%pip -q install kagglehub pandas scikit-learn matplotlib pillow

## 2 Download and inspect the data
This particular archive contains breed-named images under `images/`, rather than separate cat and dog folders. Its 37 breed names match Oxford-IIIT Pet. The code explicitly maps 12 cat breeds and 25 dog breeds to species; unknown filenames are ignored and counted. Ground truth comes from these labels, never from model predictions.

We use 200 images per species, sampled before predictions with seed 42. Corrupt images and duplicate pixels are skipped. This is a demonstration evaluation subset, not the official Oxford train/test benchmark.

In [ ]:
from pathlib import Path
import kagglehub
DATA_ROOT = Path(kagglehub.dataset_download('yashdogra/cats-and-dogs/versions/1'))
PER_CLASS = 200
SEED = 42
OUTPUT_DIR = Path('outputs')
print('Dataset:', DATA_ROOT)
print('JPEG count:', len(list(DATA_ROOT.rglob('*.jpg'))))
print('Example names:', [p.name for p in sorted(DATA_ROOT.rglob('*.jpg'))[:8]])

## 3 Imports and class labels
A **label** is the correct answer attached to an image. A **tensor** is the numeric array PyTorch operates on. ImageNet class indices 281–285 represent domestic cats; 151–268 represent domestic dog breeds. Wolves, foxes and big cats are excluded. Assertions check the mapping against Torchvision metadata.

In [ ]:
"""Pretrained ResNet-18 cat/dog inference and grouped-score Grad-CAM."""
from pathlib import Path
import argparse, hashlib, json, platform, random, re
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import torch
import torch.nn.functional as F
import torchvision
from torchvision.models import resnet18, ResNet18_Weights
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
NAMES = ['cat', 'dog']
CAT_IDS = list(range(281, 286))  # domestic cat categories only
DOG_IDS = list(range(151, 269))  # domestic dog breeds, excludes wolves/foxes
GROUPS = [CAT_IDS, DOG_IDS]
# Explicit Oxford-IIIT Pet breed mapping, verified against the supplied archive.
CAT_BREEDS = {'Abyssinian','Bengal','Birman','Bombay','British_Shorthair','Egyptian_Mau',
              'Maine_Coon','Persian','Ragdoll','Russian_Blue','Siamese','Sphynx'}
DOG_BREEDS = {'american_bulldog','american_pit_bull_terrier','basset_hound','beagle','boxer',
              'chihuahua','english_cocker_spaniel','english_setter','german_shorthaired',
              'great_pyrenees','havanese','japanese_chin','keeshond','leonberger',
              'miniature_pinscher','newfoundland','pomeranian','pug','saint_bernard',
              'samoyed','scottish_terrier','shiba_inu','staffordshire_bull_terrier',
              'wheaten_terrier','yorkshire_terrier'}




## 4 Read and sample images safely
Each image is opened, EXIF orientation is applied, and pixels are converted to RGB. Sampling is deterministic for this archive and seed. The CSV records relative filenames and pixel hashes so the selected subset can be audited.

In [ ]:
def label_for(path):
    """Accept class directories or cat.123.jpg/dog.123.jpg; never guess."""
    breed = path.stem.rsplit('_', 1)[0]
    if breed in CAT_BREEDS: return 0
    if breed in DOG_BREEDS: return 1
    aliases = {'cat': 0, 'cats': 0, 'dog': 1, 'dogs': 1}
    for parent in path.parents:
        if parent.name.lower() in aliases:
            return aliases[parent.name.lower()]
    match = re.match(r'^(cat|dog)[._\-\s]', path.name.lower())
    return aliases[match.group(1)] if match else None


def load_rgb(path):
    with Image.open(path) as im:
        return ImageOps.exif_transpose(im).convert('RGB')


def select_images(root, per_class=200, seed=SEED):
    """Random balanced subset, after excluding unreadable/duplicate files."""
    pools = {0: [], 1: []}
    unknown = 0
    for p in sorted(Path(root).rglob('*')):
        if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}:
            label = label_for(p)
            if label is None:
                unknown += 1
            else:
                pools[label].append(p)
    rng = random.Random(seed)
    selected, skipped, seen = [], [], set()
    for label in (0, 1):
        rng.shuffle(pools[label])
        count = 0
        for p in pools[label]:
            try:
                im = load_rgb(p)
                digest = hashlib.sha256(im.size.__repr__().encode() + im.tobytes()).hexdigest()
                if digest in seen:
                    skipped.append({'file': str(p.relative_to(root)), 'reason': 'duplicate pixels'})
                    continue
                seen.add(digest)
                selected.append({'path': p, 'label': label, 'sha256_pixels': digest})
                count += 1
                if count == per_class:
                    break
            except Exception as exc:
                skipped.append({'file': str(p.relative_to(root)), 'reason': str(exc)})
        if count < per_class:
            raise ValueError(f'Only {count} valid unique {NAMES[label]} images; requested {per_class}. '
                             'Check DATA_ROOT / class folder names, or reduce PER_CLASS.')
    rng.shuffle(selected)
    return selected, skipped, {'discovered': {NAMES[k]: len(v) for k,v in pools.items()},
                               'unlabeled_files_ignored': unknown}




## 5 Load ResNet and make binary predictions
**Preprocessing:** the official weight transform resizes the shorter edge to 256, center-crops 224 × 224, converts pixels to [0,1], and normalizes channels using ImageNet mean and standard deviation.

ResNet-18 already has learned weights. `eval()` selects inference behavior. `requires_grad_(False)` freezes weights; there is no optimizer, loss training loop, or weight update.

**Why group the classes?** ResNet outputs 1,000 raw scores (logits), not two. For each species, we calculate `logsumexp` over its relevant logits. Softmax of these two grouped scores equals summing ImageNet probabilities per species and renormalizing over cat + dog.

`cat_score` and `dog_score` are conditional scores under the assumption that the image is one of these animals, not calibrated reliability estimates. `cat_dog_mass` also records how much of the original 1,000-class probability belongs to these groups. A high conditional score with low mass should be treated cautiously. Unequal numbers of ImageNet classes can bias this baseline.

In [ ]:
def group_logits(logits):
    # Softmax over these logsumexp scores is equivalent to summing ImageNet
    # probabilities within each group, then conditioning on cat OR dog.
    return torch.stack([torch.logsumexp(logits[:, ids], dim=1) for ids in GROUPS], dim=1)


def build_model(device):
    weights = ResNet18_Weights.IMAGENET1K_V1
    categories = weights.meta['categories']
    assert categories[151] == 'Chihuahua' and categories[268] == 'Mexican hairless'
    assert categories[281] == 'tabby' and categories[285] == 'Egyptian cat'
    model = resnet18(weights=weights).eval().to(device)
    # Frozen weights; input requires_grad will still permit Grad-CAM.
    model.requires_grad_(False)
    return model, weights.transforms(), categories


def predict(model, preprocess, categories, selected, root, device, batch_size=32):
    rows = []
    with torch.inference_mode():
        for start in range(0, len(selected), batch_size):
            batch = selected[start:start+batch_size]
            x = torch.stack([preprocess(load_rgb(item['path'])) for item in batch]).to(device)
            logits = model(x)
            binary = group_logits(logits).softmax(1).cpu()
            all_probs = logits.softmax(1)
            mass = all_probs[:, CAT_IDS + DOG_IDS].sum(1).cpu()
            top = logits.argmax(1).cpu()
            for i, item in enumerate(batch):
                pred = int(binary[i].argmax())
                rows.append({'file': str(item['path'].relative_to(root)),
                             'true_label': NAMES[item['label']], 'predicted_label': NAMES[pred],
                             'correct': pred == item['label'], 'cat_score': float(binary[i,0]),
                             'dog_score': float(binary[i,1]), 'cat_dog_mass': float(mass[i]),
                             'imagenet_top1': categories[int(top[i])],
                             'sha256_pixels': item['sha256_pixels']})
            print(f'Predicted {min(start+batch_size, len(selected))}/{len(selected)}', flush=True)
    return pd.DataFrame(rows)




## 6 Implement Grad-CAM
A **feature map** is a grid of learned image responses. A **gradient** measures how a small change in that grid affects a target score.

We capture the last residual block output (`layer4[-1]`, 512 channels with 7 × 7 spatial resolution at this input size). For the predicted species' grouped logit, compute gradients with respect to these features. Average gradients spatially to obtain a weight per channel, take a weighted sum of feature maps, keep positive contributions with ReLU, resize to 224 × 224, and normalize to [0,1].

The heatmap explains the same species score used in prediction. Gradients are enabled only for explanation; model weights stay frozen. A zero positive map is explicitly reported. These are coarse visual explanations, not segmentation masks or proof of causation.

In [ ]:
def input_rgb(x):
    mean = torch.tensor([.485,.456,.406]).view(3,1,1)
    std = torch.tensor([.229,.224,.225]).view(3,1,1)
    return (x.detach().cpu()*std+mean).clamp(0,1).permute(1,2,0).numpy()


def gradcam(model, x, target):
    """Gradient of the chosen grouped logit at layer4[-1] block output."""
    activations = []
    handle = model.layer4[-1].register_forward_hook(lambda m, i, o: activations.append(o))
    try:
        with torch.enable_grad():
            logits = model(x.detach().clone().requires_grad_(True))
            score = group_logits(logits)[0, target]
            a = activations[0]
            g = torch.autograd.grad(score, a)[0]
            cam = (g.mean(dim=(2,3),keepdim=True)*a).sum(1,keepdim=True).relu()
            cam = F.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)[0,0]
            peak = float(cam.detach().max())
            cam = cam / peak if peak > 0 else torch.zeros_like(cam)
            return cam.detach().cpu().numpy(), peak == 0
    finally:
        handle.remove()




## 7 Create prediction galleries and aligned heatmaps
Each Grad-CAM figure shows the full original, the exact model crop, the map, and the overlay. Warm colors indicate stronger positive contribution within that image. Every map is independently normalized, so color intensities are not comparable across images.

Example selection is fixed: first two correct cats, first two correct dogs, and first two errors in the sampled ordering. If there are no errors, the gallery says so; errors are never fabricated.

In [ ]:
def save_gallery(df, root, preprocess, out, correct):
    subset = df[df.correct == correct].head(6)
    fig, axes = plt.subplots(2,3,figsize=(12,8))
    for ax in axes.ravel(): ax.axis('off')
    if subset.empty:
        axes[0,0].text(.05,.5,'No examples in this subset',fontsize=14)
    for ax, (_, row) in zip(axes.ravel(), subset.iterrows()):
        ax.imshow(input_rgb(preprocess(load_rgb(root / row.file))))
        ax.set_title(f'True: {row.true_label} | Pred: {row.predicted_label}\n'
                     f'Cat {row.cat_score:.3f} / Dog {row.dog_score:.3f}', fontsize=11)
    fig.suptitle('Correct predictions' if correct else 'Incorrect predictions',fontsize=20)
    fig.tight_layout()
    fig.savefig(out / ('correct_predictions.png' if correct else 'incorrect_predictions.png'),dpi=150)
    plt.close(fig)


def save_cam_examples(df, root, model, preprocess, device, out):
    # Fixed selection rule: first two correct per true class, then first two errors.
    choices = pd.concat([df[df.correct & (df.true_label == c)].head(2) for c in NAMES]
                        + [df[~df.correct].head(2)]).drop_duplicates('file')
    manifest = []
    for n, (_, row) in enumerate(choices.iterrows(), 1):
        original = load_rgb(root / row.file)
        x = preprocess(original)
        rgb = input_rgb(x)
        target = NAMES.index(row.predicted_label)
        cam, blank = gradcam(model, x.unsqueeze(0).to(device), target)
        overlay = np.clip(.6*rgb + .4*plt.get_cmap('jet')(cam)[...,:3],0,1)
        fig, axes = plt.subplots(1,4,figsize=(14,3.8))
        for ax, im, title in zip(axes, [original,rgb,cam,overlay],
                               ['Original image','Model input crop','Grad-CAM','Overlay on input crop']):
            if title == 'Grad-CAM':
                ax.imshow(im,cmap='jet',vmin=0,vmax=1)
            else: ax.imshow(im)
            ax.set_title(title); ax.axis('off')
        fig.suptitle(f'{row.file} | True: {row.true_label} | Predicted: {row.predicted_label}\n'
                     f'Target: grouped {row.predicted_label} score | '
                     f'Conditional score: {max(row.cat_score,row.dog_score):.3f}'
                     + (' | Zero positive activation' if blank else ''),fontsize=11)
        fig.tight_layout(rect=[0,0,1,.86])
        name = f'gradcam_{n:02d}.png'
        fig.savefig(out/name,dpi=160); plt.close(fig)
        manifest.append({'image':row.file,'figure':name,'true':row.true_label,
                         'predicted':row.predicted_label,'zero_map':blank})
    (out/'gradcam_manifest.json').write_text(json.dumps(manifest,indent=2))




## 8 Evaluate and save reproducible outputs
Accuracy is correct predictions / total. The confusion matrix uses actual labels on rows and predicted labels on columns. Precision measures how often a species prediction is right; recall measures how many actual examples of that species are found. F1 balances precision and recall.

In [ ]:
def run(root, out='outputs', per_class=200, seed=SEED, batch_size=32):
    root, out = Path(root).resolve(), Path(out)
    out.mkdir(parents=True,exist_ok=True)
    if per_class < 1 or batch_size < 1: raise ValueError('Counts must be positive.')
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.set_num_threads(min(4, torch.get_num_threads()))
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    selected, skipped, inventory = select_images(root,per_class,seed)
    model, preprocess, categories = build_model(device)
    df = predict(model,preprocess,categories,selected,root,device,batch_size)
    df.to_csv(out/'predictions.csv',index=False)
    (out/'skipped_files.json').write_text(json.dumps(skipped,indent=2))
    cm = confusion_matrix(df.true_label,df.predicted_label,labels=NAMES)
    report = classification_report(df.true_label,df.predicted_label,labels=NAMES,output_dict=True,zero_division=0)
    summary = {'dataset':'yashdogra/cats-and-dogs/versions/1', 'seed':seed,
               'per_class':per_class,'total':len(df),'correct':int(df.correct.sum()),
               'incorrect':int((~df.correct).sum()),'accuracy':float(df.correct.mean()),
               'model':'resnet18','weights':'IMAGENET1K_V1','device':str(device),
               'torch':torch.__version__,'torchvision':torchvision.__version__,
               'python':platform.python_version(),'inventory':inventory,
               'confusion_matrix':cm.tolist(),'classification_report':report}
    (out/'metrics.json').write_text(json.dumps(summary,indent=2))
    (out/'class_mapping.json').write_text(json.dumps({NAMES[i]:{str(k):categories[k] for k in ids}
                                                     for i,ids in enumerate(GROUPS)},indent=2))
    fig, ax = plt.subplots(figsize=(5,4))
    ax.imshow(cm,cmap='Blues');ax.set_xticks([0,1],NAMES);ax.set_yticks([0,1],NAMES)
    ax.set_xlabel('Predicted');ax.set_ylabel('Actual');ax.set_title(f'Confusion matrix | Accuracy {df.correct.mean():.2%}')
    for i in range(2):
        for j in range(2): ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=22,color='white' if cm[i,j]>cm.max()/2 else 'black')
    fig.tight_layout();fig.savefig(out/'confusion_matrix.png',dpi=160);plt.close(fig)
    save_gallery(df,root,preprocess,out,True);save_gallery(df,root,preprocess,out,False)
    save_cam_examples(df,root,model,preprocess,device,out)
    print(json.dumps(summary,indent=2))
    return df, summary




In [ ]:
predictions, metrics = run(DATA_ROOT, OUTPUT_DIR, PER_CLASS, SEED)
display(predictions.head(10))
print(f"Accuracy: {metrics['accuracy']:.2%}")
print(f"Correct: {metrics['correct']} | Incorrect: {metrics['incorrect']}")

## 9 Display outputs for the submission report

In [ ]:
from IPython.display import Image as DisplayImage, display
for name in ['confusion_matrix.png', 'correct_predictions.png', 'incorrect_predictions.png']:
    display(DisplayImage(filename=str(OUTPUT_DIR / name)))
for path in sorted(OUTPUT_DIR.glob('gradcam_*.png')):
    display(DisplayImage(filename=str(path)))

## 10 Interpret the actual figures
For each example, record the true label, prediction, highlighted region, and whether it mainly covers the animal or background. Compare the original and crop: did cropping remove useful evidence? For errors, inspect whether the map is diffuse, on an unusual body region, or includes background. Treat explanations of *why* an error occurred as hypotheses.

**Limitations:** forced binary prediction, unequal ImageNet group sizes, uncalibrated scores, coarse maps, center-crop information loss, a small selected sample, and possible overlap with ImageNet pretraining not audited. No claim of new training or official benchmark performance is made.

See the included report for observations from the bundled run. Recheck them if you change the sample, model, or preprocessing.

## 11 Export results and save your executed notebook

In [ ]:
import shutil
archive = shutil.make_archive('cats_dogs_results', 'zip', OUTPUT_DIR)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Results archive:', archive)
print('In Colab: File → Download → Download .ipynb to keep code and cell outputs.')

## References
- Dataset: https://www.kaggle.com/datasets/yashdogra/cats-and-dogs
- Oxford-IIIT Pet provenance: https://www.robots.ox.ac.uk/~vgg/data/pets/
- ResNet-18 weights and preprocessing: https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html
- Grad-CAM paper: https://arxiv.org/abs/1610.02391
- ResNet paper: https://arxiv.org/abs/1512.03385

Dataset images and derived output figures retain their original rights. Credit: Oxford-IIIT Pet / original image owners; Kaggle distribution by Yash Dogra. The full dataset and pretrained weights are not included in the repository.